# safe_tool

The wrapper that turns an exception into an instruction. Pure. Callers: `%run ./safe_tool`.

The instruction matters more than the message: an error that does not say what to do next is what
produces an invented number.

In [ ]:
import functools

TOOL_ERROR = (
    "ERROR in {name} ({exc_type}). This data is not available right now. Try a different tool or "
    "different arguments. If there is no alternative, tell the user this specific figure could not "
    "be retrieved - do not estimate it."
)


def safe_tool(fn):
    """Wrap a tool body so it returns the error instruction instead of raising.

    Keeps `__name__` and `__doc__`, because the docstring is what the model reads when it chooses a
    tool, and the name is what `opinion` records as the evidence `ref`.

    **Only the exception type reaches the model, never its message.** An HTTP client puts the
    request URL in the exception text, and a URL can carry a key in its query string; that text
    would then travel into the model's context, the answer and the trace. architecture.md §5.2
    forbids it and §21 records the incident that produced the rule. The type is all the model can
    act on anyway - a TimeoutError is worth retrying and a KeyError is not. If a message is ever
    needed here, it goes through the redaction filter §5.2 prescribes, not raw.
    """
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            return TOOL_ERROR.format(name=getattr(fn, "__name__", "tool"),
                                     exc_type=type(exc).__name__)

    return wrapper